# Ćwiczenie: IPW krok po kroku (MLE i kalibrowany)

**Czas**: ~90 minut | **Dane**: `admin` (próba nielosowa CBOP) + `jvs` (próba losowa) | **Cel**: od estymatora naiwnego do kalibrowanego IPW w 4 krokach

W każdej części znajdziesz **przykładowy kod** — Twoim zadaniem jest go **zmodyfikować i rozszerzyć**.

# Część A: Dane i estymator naiwny

## Pakiety

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import fsolve
from scipy.special import expit

## Wczytanie danych

In [ ]:
admin = pd.read_csv("../data/admin.csv")
jvs = pd.read_csv("../data/jvs.csv")

## region jako tekst z wiodacym zerem (zgodnosc z R)
admin["region"] = admin["region"].astype(str).str.zfill(2)
jvs["region"] = jvs["region"].astype(str).str.zfill(2)

## wielkosc populacji estymowana z JVS
N_pop = jvs["weight"].sum()

print("admin (proba nielosowa S_A):")
print(admin.head())
print(f"\njvs (proba losowa S_B):")
print(jvs.head())

Dane `admin` zawierają 9344 firm z rejestru CBOP (próba nielosowa). Dane `jvs` zawierają 6523 firm z badania popytu na pracę (próba losowa ze znanymi wagami). Zmienne wspólne: `private`, `size`, `nace`, `region`. Zmienna celu (tylko w admin): `single_shift`.

### Zadanie A1

Porównaj rozkład zmiennej `size` w obu źródłach danych. Dla JVS pamiętaj o wagach!

In [ ]:
## Tutaj kod -- porównanie rozkładu size w admin vs JVS

**Odpowiedź:** *...*

### Zadanie A2

Oblicz naiwną średnią zmiennej `single_shift` z próby admin (bez żadnych korekt). Zapisz wynik — będziesz go porównywać z estymatorami IPW w kolejnych częściach.

**Podpowiedź**: `admin["single_shift"].mean()`

In [ ]:
## Tutaj kod

**Odpowiedź:** *...*

# Czesc B: Estymator IPW-MLE

Model selekcji: $P(R_A = 1 \mid \boldsymbol{x}) = \pi(\boldsymbol{x}, \boldsymbol{\gamma})$. Wagi IPW: $w_i = 1/\hat{\pi}(\boldsymbol{x}_i)$.

Estymator IPW 1 (Horvitza-Thompsona): $\hat{\mu}_{IPW1} = \frac{\sum w_i y_i}{\hat{N}}$, gdzie $\hat{N} = \sum_{i \in S_B} d_i^B$.

> **Uwaga**: w tym ćwiczeniu stosujemy **estymator IPW 1** z $\hat{N}$ w mianowniku (tak domyślnie działa `nonprob()` w R). Estymator IPW 2 (Hajeka) z $\sum w_i$ w mianowniku omówimy na kolejnych zajęciach.

## Przyklad: model z jedna zmienna (`size`)

Pseudo-MLE: rozwiazujemy rownanie score:

$$\boldsymbol{U}(\boldsymbol{\gamma}) = \sum_{i \in S_A} \boldsymbol{x}_i - \sum_{i \in S_B} d_i^B \pi(\boldsymbol{x}_i, \boldsymbol{\gamma}) \boldsymbol{x}_i = \boldsymbol{0}$$

Ponizej gotowy kod dla prostego modelu $P(R_A = 1 \mid \text{size})$. Uruchom go i przeanalizuj wyniki.

In [ ]:
## Pseudo-MLE: model z jedna zmienna (size)
formula_vars_simple = ["size"]
cat_vars_simple = ["size"]

admin_dum_s = pd.get_dummies(admin[formula_vars_simple], columns=cat_vars_simple,
                              dtype=float, drop_first=True)
jvs_dum_s = pd.get_dummies(jvs[formula_vars_simple], columns=cat_vars_simple,
                            dtype=float, drop_first=True)
all_cols_s = sorted(set(admin_dum_s.columns) | set(jvs_dum_s.columns))
admin_dum_s = admin_dum_s.reindex(columns=all_cols_s, fill_value=0)
jvs_dum_s = jvs_dum_s.reindex(columns=all_cols_s, fill_value=0)

X_admin_s = sm.add_constant(admin_dum_s).values.astype(float)
X_jvs_s = sm.add_constant(jvs_dum_s).values.astype(float)
w_jvs_s = jvs["weight"].values

def score_eq_simple(gamma):
    pi_jvs = expit(X_jvs_s @ gamma)
    return X_admin_s.sum(axis=0) - (X_jvs_s * (w_jvs_s * pi_jvs)[:, None]).sum(axis=0)

gamma_hat_simple = fsolve(score_eq_simple, np.zeros(X_admin_s.shape[1]))
ps_simple = expit(X_admin_s @ gamma_hat_simple)
w_simple = 1.0 / ps_simple

## estymator HT
mu_simple = np.sum(w_simple * admin["single_shift"].values) / N_pop
print(f"Oszacowanie IPW-MLE (~size): {mu_simple:.4f}")
print(f"\nRozklad wag:")
print(pd.Series(w_simple).describe().round(2))

### Zadanie B1

Zmodyfikuj powyzszy kod tak, aby model selekcji wykorzystywal **wszystkie zmienne**: `private, size, nace, region`.

**Podpowiedz**: zmien `formula_vars` na `["private", "size", "nace", "region"]` i `cat_vars` na `["size", "nace", "region"]` (zmienna `private` jest numeryczna).

In [ ]:
## Tutaj kod -- IPW-MLE z wszystkimi zmiennymi (pseudo-MLE)
## Skopiuj kod z przykladu i zmien formula_vars i cat_vars

### Zadanie B2

Dla modelu pełnego:

a) Wypisz rozkład wag (`pd.Series(w).describe()`). Jaka jest min/max waga?

b) Sprawdź balans zmiennej `size` używając poniższej funkcji:

In [ ]:
## funkcja do sprawdzania balansu
def check_balance(admin, jvs, var, ipw_weights):
    cats = sorted(admin[var].unique())
    rows = []
    for c in cats:
        rows.append({
            "Kategoria": c,
            "CBOP (raw)": round((admin[var] == c).mean(), 4),
            "CBOP (IPW)": round(np.average(admin[var] == c, weights=ipw_weights), 4),
            "JVS (ważone)": round(np.average(jvs[var] == c, weights=jvs["weight"]), 4)
        })
    return pd.DataFrame(rows)

## Tutaj kod -- użyj check_balance() z wagami z Twojego modelu

**Odpowiedź:** *...*

# Część C: Kalibrowany estymator IPW (GEE)

MLE nie gwarantuje odtworzenia sum populacyjnych. GEE rozwiązuje równanie kalibracyjne:

$$\boldsymbol{G}(\boldsymbol{\gamma}) = \sum_{i \in S_A} \frac{\boldsymbol{x}_i}{\pi(\boldsymbol{x}_i, \boldsymbol{\gamma})} - \sum_{i \in S_B} d_i^B \boldsymbol{x}_i = \boldsymbol{0}$$

### Zadanie C1

Uzupełnij poniższy szablon (wstaw listę zmiennych w `formula_vars`) i uruchom model GEE:

In [ ]:
## Ponizej szablon GEE -- uzupelnij formula_vars (wstaw liste zmiennych)
## i uruchom caly blok

## formula_vars = ["private", "size", "nace", "region"]  ## <-- odkomentuj i uzupelnij
## cat_vars = ["size", "nace", "region"]

## admin_dum = pd.get_dummies(admin[formula_vars],
##                             columns=cat_vars,
##                             dtype=float, drop_first=True)
## jvs_dum = pd.get_dummies(jvs[formula_vars],
##                           columns=cat_vars,
##                           dtype=float, drop_first=True)
## all_cols = sorted(set(admin_dum.columns) | set(jvs_dum.columns))
## admin_dum = admin_dum.reindex(columns=all_cols, fill_value=0)
## jvs_dum = jvs_dum.reindex(columns=all_cols, fill_value=0)
##
## X_admin = sm.add_constant(admin_dum).values
## X_jvs = sm.add_constant(jvs_dum).values
## w_jvs = jvs["weight"].values
##
## tau_x = (X_jvs * w_jvs[:, None]).sum(axis=0)
##
## def gee_equations(gamma):
##     pi_admin = np.clip(expit(X_admin @ gamma), 1e-10, 1 - 1e-10)
##     return (X_admin / pi_admin[:, None]).sum(axis=0) - tau_x
##
## gamma0 = np.zeros(X_admin.shape[1])
## gamma_gee = fsolve(gee_equations, gamma0)
## ps_gee = expit(X_admin @ gamma_gee)
## w_gee = 1.0 / ps_gee
##
## mu_gee = np.sum(w_gee * admin["single_shift"].values) / N_pop
## print(f"Oszacowanie IPW-GEE: {mu_gee:.4f}")

### Zadanie C2

Porównaj wyniki MLE i GEE:

a) Wypisz oszacowania obu metod. Czy się różnią?

b) Sprawdź balans zmiennej `size` dla obu metod.

c) Narysuj wykres porównujący wagi MLE vs GEE (`plt.scatter(w_mle, w_gee)`).

In [ ]:
## Tutaj kod -- porównanie MLE vs GEE

**Odpowiedź:** *...*

### Zadanie C3

GEE z wartościami globalnymi — gdy nie mamy danych jednostkowych z JVS, ale znamy sumy populacyjne:

In [ ]:
pop_totals = {"const": 51870, "size_M": 13758, "size_S": 29551}

admin_dum_s = pd.get_dummies(admin[["size"]], columns=["size"],
                              dtype=float, drop_first=True)
X_admin_s = sm.add_constant(admin_dum_s).values
col_names_s = ["const"] + sorted(admin_dum_s.columns.tolist())
tau_x_s = np.array([pop_totals.get(c, 0) for c in col_names_s])

def gee_eq_totals(gamma):
    pi_a = np.clip(expit(X_admin_s @ gamma), 1e-10, 1 - 1e-10)
    return (X_admin_s / pi_a[:, None]).sum(axis=0) - tau_x_s

result_tot = fsolve(gee_eq_totals, np.zeros(X_admin_s.shape[1]))
ps_tot = expit(X_admin_s @ result_tot)
mu_gee_tot = np.sum((1.0 / ps_tot) * admin["single_shift"].values) / N_pop
print(f"IPW-GEE (wartosci globalne): {mu_gee_tot:.4f}")

**Odpowiedź:** *...*

# Część D: Podsumowanie

### Zadanie D1

Uzupełnij tabelę swoimi wynikami:

| Estymator | Oszacowanie $\hat{\mu}$ |
|-----------|------------------------|
| Naiwny | *...* |
| IPW-MLE (~size) | *...* |
| IPW-MLE (pełny) | *...* |
| IPW-GEE (pełny) | *...* |
| IPW-GEE (pop_totals) | *...* |

### Zadanie D2

a) Jak zmieniły się oszacowania po zastosowaniu IPW w porównaniu z estymatorem naiwnym? W którą stronę?

b) W jakich sytuacjach praktycznych estymator GEE jest lepszy od MLE?

c) Kiedy w praktyce użyjesz `pop_totals` zamiast danych jednostkowych?

**Odpowiedź:** *...*